In [11]:
script_name = "021_2023_TW_Estimation"

# Project Setup

In [12]:
from pathlib import Path

# Assume this script is located in 02_code/
project_dir = Path("..").resolve()

# Public data directory
data_dir = project_dir / "01_data"

# Output base directory
output_dir = project_dir / "outputs"
output_dir.mkdir(exist_ok=True)


# Script-specific output directories
midstream_dir = output_dir / script_name / "midstream"
results_dir   = output_dir / script_name / "results"

# Create directories
midstream_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

In [13]:
seed = 0

# Import Libraries

In [14]:
# =====================================================
#  Environment settings
# =====================================================
import os
os.environ["OMP_NUM_THREADS"] = "1"  

# =====================================================
#  Standard library
# =====================================================
import csv
import datetime as dt
import inspect
import itertools
import pickle
import random
import string
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN


# =====================================================
#  Third-party libraries
# =====================================================
import pandas as pd
from pandas.api.types import CategoricalDtype

from tqdm import tqdm

import matplotlib.pyplot as plt


# =====================================================
#  Scikit-learn
# =====================================================
# Model selection
from sklearn.model_selection import KFold, GridSearchCV

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Feature selection
from sklearn.feature_selection import (
    SelectKBest,
    SelectPercentile,
    SelectFromModel,
)

# Models
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import mean_squared_error, r2_score

# Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

# Pipeline
from sklearn.pipeline import Pipeline, make_pipeline


# =====================================================
#  Utility functions
# =====================================================
def remove_exponent(d):
    return d.quantize(Decimal(1)) if d == d.to_integral() else d.normalize()

# Data Loading and Preprocessing

In [15]:
df = pd.read_csv(data_dir / "FT_2023_df.csv", index_col = 0)

df["HarvTime"] = df["HarvTime"].astype(str)
df = df.rename(columns={"PlotID": "ID4human"})

df

,ID4human,Origin,Density,block,HarvTime,Wei_all_per_m2,PH_max_202304171517,PH_90_202304171517,PH_max_202305031115,PH_90_202305031115,...,GNDVI_median_202305031115,GNDVI_median_202305181131,GNDVI_median_202305301127,GNDVI_median_202306121122,GNDVI_median_202306271027,NDRE_median_202305031115,NDRE_median_202305181131,NDRE_median_202305301127,NDRE_median_202306121122,NDRE_median_202306271027
0,T1-02_2_00,Memanbetsu,sparse,T1,2,1360.799001,0.000000,0.000000,0.177719,0.122615,...,0.636364,0.686275,0.705882,0.500000,0.444444,0.084746,0.072165,0.128205,0.166667,0.142857
3,T2-13_1_00,Memanbetsu,sparse,T2,3,2681.647940,0.000000,0.000000,0.215233,0.141205,...,0.640000,0.718310,0.722222,0.692308,0.500000,0.070423,0.150685,0.155556,0.147059,0.125000
4,T1-35_2_00,Memanbetsu,sparse,T1,3,1777.777778,0.000000,0.000000,0.192322,0.123990,...,0.631579,0.692308,0.696970,0.666667,0.600000,0.074074,0.142857,0.090909,0.150000,0.111111
6,T2-24_2_00,Memanbetsu,sparse,T2,4,2476.903870,0.000000,0.000000,0.179230,0.133919,...,0.658537,0.698113,0.703704,0.673469,0.636364,0.063291,0.137931,0.162791,0.133333,0.130435
7,T1-22_1_00,Tokachi,sparse,T1,4,2219.225968,0.000000,0.000000,0.172813,0.127785,...,0.672727,0.680000,0.677419,0.689655,0.650000,0.152174,0.126761,0.130435,0.171429,0.153846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368,B-7-3,Aomori,standard,M3,5,3338.271605,0.000000,0.000000,0.157768,0.116830,...,0.632653,0.680000,0.692308,0.695652,0.666667,0.058824,0.116279,0.136842,0.161905,0.147541
371,D-8-2,Tokachi,dense,M4,5,3333.333333,0.000000,0.000000,0.183426,0.129199,...,0.647059,0.702128,0.714286,0.701493,0.614035,0.068966,0.135802,0.175258,0.164179,0.123288
372,B-7-2,Aomori,standard,M3,5,3098.765432,0.016533,0.015445,0.178909,0.124451,...,0.629630,0.684211,0.696970,0.687500,0.647059,0.058824,0.116883,0.139241,0.160000,0.138889
373,B-7-1,Aomori,standard,M3,5,2597.530864,0.021011,0.019299,0.171104,0.116610,...,0.636364,0.680000,0.687500,0.681818,0.629630,0.061224,0.118644,0.130435,0.157895,0.136364


In [16]:
year_now = "2023"
# Specify when to use the trained model set
model_timestamp_config_df = pd.read_csv(data_dir / "0_model_timestamp_config.csv", index_col=0, dtype=str).apply(lambda col: col.str.strip('"'))

CV_modeltime_now = model_timestamp_config_df.loc["RandomCV", year_now]

# Utility Functions

In [17]:
def df_formatting(df4def, Label_names):
    df4def.loc[:,Label_names] = df4def.loc[:,Label_names].astype(str)
    
    Orig_type = CategoricalDtype(categories=['Douo' ,'Memanbetsu','Tokachi','Aomori'], ordered=True)
    Dens_type = CategoricalDtype(categories=['sparse' ,'standard','dense'], ordered=True)
    df4def.loc[:,"Origin"] = df4def.loc[:,"Origin"].astype(Orig_type)
    df4def.loc[:,"Density"] = df4def.loc[:,"Density"].astype(Dens_type)
    
    temp_df = df4def[Label_names] 
    df4def= df4def.drop(Label_names, axis= 1)
    for i in reversed(range(len(Label_names))):
        df4def.insert(0, Label_names[i], temp_df[Label_names[i]])

    return df4def

In [18]:
def tra_tes_plot(y_train, y_train_pred, y_test, y_test_pred, ymin, ymax):
    fig_num = 2
    fig,axes = plt.subplots(nrows=1,ncols=fig_num, figsize=(fig_num*4,4)) 
    for i4 in range(fig_num):
        if i4 ==0:
            axes[i4].scatter(
                y_train,  
                y_train_pred,  
                c='steelblue',
                edgecolor='white',
                marker='o', 
                s=35,
                alpha=0.9,
                label='Training data')
            r_train = pd.Series(y_train.to_list()).corr(pd.Series(y_train_pred.tolist()))
            R2_train = r2_score(y_train, y_train_pred)
            RMSE_train = mean_squared_error(y_train, y_train_pred, squared=False)
            RMSE_train_rounded = float(remove_exponent(Decimal(RMSE_train).quantize(Decimal("0.01"), ROUND_HALF_UP)))
            axes[i4].text(0.60,0.12,r'$r$: %.2f' % (r_train), transform=axes[i4].transAxes)
            axes[i4].text(0.60,0.08,r'$RMSE$: %.2f' % (RMSE_train_rounded), transform=axes[i4].transAxes)
            axes[i4].text(0.60,0.04,r'$R^2$: %.2f' % R2_train,transform=axes[i4].transAxes)
            axes[i4].set_title(("Training")) 

        elif i4 ==1:
            axes[i4].scatter(
                y_test,  
                y_test_pred,   
                c='limegreen',
                edgecolor='white',
                marker='s', 
                s=35,
                alpha=0.9,
                label='Test data')
            r_test = pd.Series(y_test.to_list()).corr(pd.Series(y_test_pred.tolist()))
            R2_test = r2_score(y_test, y_test_pred)
            RMSE_test = mean_squared_error(y_test, y_test_pred, squared=False)
            RMSE_test_rounded = float(remove_exponent(Decimal(RMSE_test).quantize(Decimal("0.01"), ROUND_HALF_UP)))
            axes[i4].text(0.60,0.12,r'$r$: %.2f' % (r_test), transform=axes[i4].transAxes)
            axes[i4].text(0.60,0.08,r'$RMSE$: %.2f' % (RMSE_test_rounded), transform=axes[i4].transAxes)
            axes[i4].text(0.60,0.04,r'$R^2$: %.2f' % R2_test, transform=axes[i4].transAxes)
            axes[i4].set_title(("Test"))

        axes[i4].set_xlabel('Observed values')
        axes[i4].set_ylabel('Predicted values')


        plo_x = np.linspace(ymin,ymax,(ymax-ymin)) 
        axes[i4].plot(plo_x, plo_x, color="black", linestyle = ":") # y = x

        axes[i4].set_xlim([ymin, ymax])
        axes[i4].set_ylim([ymin, ymax])

        axes[i4].set_aspect("equal", adjustable="box")


    plt.subplots_adjust(wspace=0.5)
    fig.tight_layout()
    
    return fig    

def all_test_dataframe(pred_df, CV_alpha_li, harv_now, rep_now, target_name):
    for alp_now in CV_alpha_li:

        x = pred_df.loc[pred_df["Group" + str(harv_now) + "_" + str(rep_now) ] == alp_now, 
                        target_name]
        y = pred_df.loc[pred_df["Group" + str(harv_now) + "_" + str(rep_now) ] == alp_now,
                        [s for s in pred_df.columns if s.endswith("Rep-" + str(rep_now) + "_TestGroup-" + alp_now + "_pred")][0]] 
        if alp_now == "A":
            x_all = x
            y_all = y
        else:

            x_all = pd.concat([x_all, x])
            y_all = pd.concat([y_all, y])
            
    return x_all, y_all

def all_test_plot(x_all, y_all,tar_min, tar_max):
    r = pd.Series(x_all.to_list()).corr(pd.Series(y_all.tolist()))
    R2 = r2_score(x_all, y_all)
    RMSE = mean_squared_error(x_all, y_all, squared=False)
    RMSE_rounded = float(remove_exponent(Decimal(RMSE).quantize(Decimal("0.01"), ROUND_HALF_UP)))
            
    fig_num = 1
    fig,ax = plt.subplots(nrows=1,ncols=fig_num, figsize=(fig_num*4,4))
    
    ax.scatter(x_all,  
                y_all,  
                c='steelblue',
                edgecolor='white',
                marker='o', 
                s=35,
                alpha=0.9)

    ax.text(0.60,0.12,r'$r$: %.2f' % (r), transform=ax.transAxes)
    ax.text(0.60,0.08,r'$RMSE$: %.2f' % (RMSE_rounded), transform=ax.transAxes)
    ax.text(0.60,0.04,r'$R^2$: %.2f' % R2,transform=ax.transAxes)
    ax.set_title(("10fold-CV Test")) # 

    plo_x = range(tar_min, tar_max)
    ax.plot(plo_x, plo_x, color="black", linestyle = ":")
    
    return fig

# 10-Fold Cross-Validation for TW Estimation

In [19]:
def K10foldCVwithPipe5(time_now, rep_num, harv_now, model_name_li, df, target_name, seed, latest_flight_date_li,
                       demo = False, do_cv=True, do_savefig=True):   
    # Setting Parameters    
    strategy_now = inspect.currentframe().f_code.co_name
    ran_num = 0
    ii = 0 
    target_min_max_li = [[0],[8000]] 
    target_HaT_min_max_li = [[0, 0,1000,2000,3000,4000], 
                             [1000, 1500,3000, 5500, 7000, 8000]] 
    
    # data frame containing only final harvest 
    HarvTime_now = int(harv_now)
    harv_now_df = df[df["HarvTime"] == str(HarvTime_now)].copy()
    harv_now_df.reset_index(drop=True, inplace=True)

    # Creating a table data  to store the results of model performance evaluation
    model_eva_li = ["r", "R2", "RMSE"] 
    resu_df_col_names = ["model_fullname", "HarvTime", "Flight_date", "img_type", "samp_rate", "samp_method", "rep"] + model_eva_li
    resu_df = pd.DataFrame(index = [], columns=resu_df_col_names) 

    # set save dir
    str_dt_now = time_now 
    if demo == True:
        base_model_dir = midstream_dir / f"CV10_harv{harv_now}" / f"{strategy_now}_{str_dt_now}" 
    else:
        base_model_dir = results_dir / f"CV10_harv{harv_now}" / f"{strategy_now}_{str_dt_now}" 
    os.makedirs(base_model_dir, exist_ok=True)

    # Flight Combination Settings
    latest_flight_date = latest_flight_date_li[HarvTime_now]
    CC_m2_cols_li = df.columns[df.columns.str.contains("CC_m2_")].to_list() 
    flight_date_li = [s.split("CC_m2_")[1] for s in CC_m2_cols_li] 
    NDRE_median_cols_li = df.columns[df.columns.str.contains("NDRE_median_")].to_list() 
    mul_flight_date_li = [s.split("NDRE_median_")[1] for s in NDRE_median_cols_li] 
    before_flight_datetime_li = [s for s in flight_date_li if s[:8] in (latest_flight_date_li[:(HarvTime_now+1)])] 
    input_num_li = [list(itertools.combinations([i for i in range(0,HarvTime_now+1)],s))for s in range(1, HarvTime_now+2)] 
    input_num_li = [s1 for s2 in input_num_li for s1 in s2] 
    
    # Setting Sampling ratio 
    gap = 25 
    per_li = [s for s in range(gap,100+gap,gap)] 
    
    for input_num_now, img_type, per_i, ran_or_kme in itertools.product(range(len(input_num_li)),["RGB", "Full"], range(len(per_li)),["random", "kmeans"]):
        # check
        if demo == True:
            if img_type != "RGB":
                continue

            if ran_or_kme != "kmeans": 
                continue

        input_num_name_now = "".join([str(s) for s in list(input_num_li[input_num_now])]) 
        input_comb_date_now = [latest_flight_date_li[s] for s in list(input_num_li[input_num_now])] 
        input_comb_now = [s[:8] for s in flight_date_li ] 

        # Setting the learning Scenario
        RGB_input_comb_now = [s for s in flight_date_li if s[:8] in (input_comb_date_now)]
        RGB_input_UAV_cols = [[ s1 + "_" + s for s1 in RGB_Pheno_names] for s in RGB_input_comb_now] 
        RGB_input_UAV_cols = [x for row in RGB_input_UAV_cols for x in row] 
        
        mul_input_comb_now = [s for s in mul_flight_date_li if s[:8] in (input_comb_date_now)]
        mul_input_UAV_cols = [[ s1 + "_" + s for s1 in  Mul_Pheno_names] for s in mul_input_comb_now] 
        mul_input_UAV_cols = [x for row in mul_input_UAV_cols for x in row]
        
        if img_type == "RGB": 
            input_UAV_cols = RGB_input_UAV_cols 

        elif img_type == "Multi": 
            input_UAV_cols = mul_input_UAV_cols 

        elif img_type == "Full": 
            input_UAV_cols = RGB_input_UAV_cols + mul_input_UAV_cols 
            
        # Replication 
        ran_num +=1
        for rep_now in range(1, rep_num + 1): 
            pred_df = df.copy() 
            proto_pred_df = df.copy() 

            # Printing the model's basic imformation
            print(input_num_now, img_type, per_li[per_i], ran_or_kme, rep_now)

            # Setting seed
            random.seed(seed)
            rep_dict = dict(zip([str(s) for s in range(1,rep_num + 1)], random.sample(range(1,100), k = rep_num))) 

            ## 10-fold CV
            CV_k = 10
            kf = KFold(n_splits = CV_k, shuffle = True, random_state=rep_dict[str(rep_now)])
            train_group_index_li = []
            test_group_index_li = []
            
            for train_i, test_i in kf.split(harv_now_df):
                train_group_index_li += [train_i.tolist()]
                test_group_index_li += [test_i.tolist()]
            CV_alpha_li = list(string.ascii_uppercase[0:CV_k])
            for group_num_now, group_alp_now in zip(range(0,CV_k),CV_alpha_li ): 
                harv_now_df.loc[test_group_index_li[group_num_now], "Group" + str(harv_now) + "_" + str(rep_now)] = group_alp_now

            ## Data frame for training model
            df_now = harv_now_df[Label_names + ["Group" + str(harv_now) + "_" + str(rep_now)] + input_UAV_cols + [target_name]] 
            X = df_now[Label_names + ["Group" + str(harv_now) + "_" + str(rep_now)] + input_UAV_cols]
            y = df_now[Label_names + ["Group" + str(harv_now) + "_" + str(rep_now)] + [target_name]]
            if demo == True: print(f"input_UAV_cols is {input_UAV_cols}") 
            if demo == True: print(f"target_name is {target_name}")
                
            ## Data frame for estimation
            pheno_df_now = pred_df[pred_df["HarvTime"] >= str(HarvTime_now)][Label_names + input_UAV_cols + [target_name]]  
            base_X_pheno = pheno_df_now[input_UAV_cols]

            y_pheno = pheno_df_now[target_name]
            
            # ============================================
            # Model Training Methods and Parameter Settings
            # ============================================
            
            ## Setting the regressor
            regressor_dic = {"RF":RandomForestRegressor(random_state=rep_dict[str(rep_now)])}
            ## Feature Selection
            ft_sele_now = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state= rep_dict[str(rep_now)]),
                                          threshold="median")
            ## Pipeline
            pipe = Pipeline([("scaler", StandardScaler()), 
                             ("feature_selection",ft_sele_now),
                             ("regressor",[regressor_dic["RF"]])]) 

            param_grid = [{"regressor":[regressor_dic['RF']], 
                 "scaler":[StandardScaler()], 
                 "feature_selection":[ft_sele_now],
                 "regressor__n_estimators":[10,100,1000],
                 "regressor__max_depth":[5,10,15, 20, None]}]

            ## Setting the model name 
            model_fullname_li = ["Harv-" + str(HarvTime_now),
                                 "Model-" + "RF",                                 
                                 "UAVinput-"+ input_num_name_now, 
                                 "ImgType-" + img_type, 
                                 "SampRate-" + str(per_li[per_i]),
                                 "SampMetho-" + ran_or_kme, 
                                 "Rep-" + str(rep_now)]
            model_fullname_ori = "_".join(model_fullname_li)
            
            # ============================================
            # Prototype model
            # ============================================
            proto_model_fullname = "proto_" + "bloCV-Non" + "_" + model_fullname_ori
            
            ## Training data for prototype model
            proto_X_train = X[input_UAV_cols]
            proto_y_train = y[target_name]

            tra_rate = (CV_k-1)/CV_k
            
            # Checking the strategy of sampling
            if ran_or_kme == "random": # Random sampling
                proto_X_train = proto_X_train.sample(frac=(tra_rate),random_state = ran_num).sort_index()
                X_train = proto_X_train.sample(frac=(per_li[per_i]/100), random_state = ran_num).sort_index()
                y_train = proto_y_train[X_train.index.values.tolist()].sort_index()

            elif ran_or_kme == "kmeans": # Diversity Sampling (k-means)
                proto_X_train = proto_X_train.sample(frac=(tra_rate),random_state = ran_num+1).sort_index()
                kmeans = KMeans(n_clusters=int((len(proto_X_train )*(per_li[per_i]/100))),
                                random_state= (ran_num + 1), 
                                n_init="auto")

                _ = kmeans.fit(proto_X_train)

                closest, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, proto_X_train)
                X_train =  proto_X_train.iloc[closest].sort_index()
                y_train = proto_y_train[X_train.index.values.tolist()].sort_index()

            ## Traing the prototype model
            proto_grid = GridSearchCV(pipe, param_grid,scoring = "neg_root_mean_squared_error",
                                      cv=4, n_jobs= 10, verbose=0) 
            _ = proto_grid.fit(X_train, y_train) ; 

            with open(base_model_dir / f"{proto_model_fullname}.pickele", mode="wb") as f: 
                pickle.dump(proto_grid, f)

            with open(base_model_dir / f"{proto_model_fullname}.pickele", mode="rb") as f:
                proto_grid = pickle.load(f)

            ## Estimation of proto model
            proto_pred_df = df.copy()
            proto_y_pheno_pred_df = pheno_df_now.copy() 
            proto_y_pheno_pred = proto_grid.predict(base_X_pheno) 
            proto_y_pheno_pred_df[proto_model_fullname + "_pred"] = proto_y_pheno_pred
            proto_pred_df = pd.merge(proto_pred_df, proto_y_pheno_pred_df, how="left") 
            proto_pred_df["pred_DAS"] = samp_DAS_dic[harv_now]
            proto_pred_df.to_csv(base_model_dir / f"pred_df_{proto_model_fullname}.csv")
            proto_pred_df = pd.read_csv(base_model_dir / f"pred_df_{proto_model_fullname}.csv", index_col=0)
            proto_pred_df = df_formatting(proto_pred_df, Label_names)
            
            # ============================================
            # 10-fold CV
            # ============================================
            ## Adjust the number of training data
            for CV_now in (CV_alpha_li): 
                train_CVgroup_now = [s for s in CV_alpha_li if s != CV_now]
                test_CVgroup_now = [CV_now]

                ## Separate X
                base_X_train = X.loc[X["Group" + str(harv_now) + "_" + str(rep_now)].isin(train_CVgroup_now),input_UAV_cols]
                base_X_test = X.loc[X["Group" + str(harv_now) + "_" + str(rep_now)].isin(test_CVgroup_now),input_UAV_cols]

                ## Separate Y
                base_y_train = y.loc[y["Group" + str(harv_now) + "_" + str(rep_now)].isin(train_CVgroup_now),target_name]
                y_test = y.loc[y["Group" + str(harv_now) + "_" + str(rep_now)].isin(test_CVgroup_now),target_name]

                seed_4samp = int((rep_dict[str(rep_now)] + 100) * per_li[per_i]) + ran_num

                ## Cheking of sampling strategy
                if ran_or_kme == "random":
                    X_train = base_X_train.sample(frac=(per_li[per_i]/100), random_state = seed_4samp).sort_index()
                    y_train = base_y_train[X_train.index.values.tolist()].sort_index()

                elif ran_or_kme == "kmeans":
                    kmeans = KMeans(n_clusters=int((len(base_X_train )*(per_li[per_i]/100))),
                                    random_state= (seed_4samp + 1),
                                    n_init="auto")

                    _ = kmeans.fit(base_X_train)

                    closest, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, base_X_train)
                    X_train =  base_X_train.iloc[closest].sort_index()
                    y_train = base_y_train[X_train.index.values.tolist()].sort_index()

                X_test = base_X_test  
                y_test = y_test 

                model_fullname = model_fullname_ori + "_" + "TestGroup-" + CV_now 

                ## Train the model
                if do_cv == True:
                    grid = GridSearchCV(pipe, param_grid, scoring = "neg_root_mean_squared_error", cv=4, n_jobs= 10, verbose=0) 
                    _ = grid.fit(X_train, y_train) ; 

                    with open(base_model_dir /f"{model_fullname}.pickele", mode="wb") as f: 
                        pickle.dump(grid, f)

                    with open(base_model_dir /f"{model_fullname}.pickele", mode="rb") as f: 
                        grid = pickle.load(f)

                    ## Saving estimated values 
                    y_pheno_pred_df = pheno_df_now.copy() 
                    y_pheno_pred = grid.predict(base_X_pheno) 
                    y_pheno_pred_df[model_fullname + "_pred"] = y_pheno_pred 

                    pred_df = pd.merge(pred_df, y_pheno_pred_df, how="left") 

                    ## Estimation
                    y_train_pred = grid.predict(X_train) 
                    y_test_pred = grid.predict(X_test) 
                    output_ndim = y_train_pred.ndim 
                    if output_ndim == 2: 
                        y_train_pred = np.array([s1 for s2 in y_train_pred.tolist() for s1 in s2])
                        y_test_pred = np.array([s1 for s2 in y_test_pred.tolist() for s1 in s2])

                    ymin_now = target_min_max_li[0][i_t] 
                    ymax_now = target_min_max_li[1][i_t] 

                    CV_train_df = pd.DataFrame(
                        {"y_train":y_train,
                         "y_train_pred":y_train_pred})
                    CV_train_df.to_csv(base_model_dir / f"train_df_{model_fullname}.csv")
                    CV_train_df = pd.read_csv(base_model_dir / f"train_df_{model_fullname}.csv", index_col=0)

                    CV_test_df = pd.DataFrame(
                        {"y_test":y_test,
                         "y_test_pred":y_test_pred})
                    CV_test_df.to_csv(base_model_dir / f"test_df_{model_fullname}.csv")
                    CV_test_df = pd.read_csv(base_model_dir / f"test_df_{model_fullname}.csv", index_col=0)


            # Summary of Model Evaluation Using CV
            if do_cv == True:   
                key_df = df_now.loc[:, ["ID4human", "Group" + str(harv_now) + "_" + str(rep_now)]]  
                pred_df = pd.merge(pred_df, key_df, how = "left", on = "ID4human")
                pred_df.to_csv(base_model_dir / f"pred_df_f{model_fullname_ori}.csv") 
                pred_df = pd.read_csv(base_model_dir / f"pred_df_f{model_fullname_ori}.csv", index_col=0)
                pred_df = df_formatting(pred_df, Label_names)

                x_all, y_all = all_test_dataframe(pred_df,CV_alpha_li, harv_now,rep_now, target_name)
                if do_savefig == True:
                    fig = all_test_plot(x_all, y_all,target_HaT_min_max_li[0][HarvTime_now-1],target_HaT_min_max_li[1][HarvTime_now-1])
                    fig.savefig(base_model_dir / f"scatplo_{model_fullname_ori}.png")
                    plt.show(fig)
                    plt.close(fig)

                ## Creating a data frame of model evaluation metrics
                r = pd.Series(x_all.to_list()).corr(pd.Series(y_all.tolist()))
                R2 = r2_score(x_all, y_all)
                RMSE = mean_squared_error(x_all, y_all, squared=False)
                RMSE_rounded = float(remove_exponent(Decimal(RMSE).quantize(Decimal("0.01"), ROUND_HALF_UP)))

                resu_li_now = [model_fullname, HarvTime_now, input_num_now, img_type, per_li[per_i], ran_or_kme, rep_now] 
                resu_li_now = resu_li_now + [r, R2, RMSE] 
                resu_df.loc[ii] = resu_li_now 
            ii += 1
    resu_df.to_csv(base_model_dir / f"resu_df_{harv_now}.csv" )
    
    if do_cv == False:
        resu_df = "None"
    return resu_df 

## Model Parameters

In [20]:
# UAV flight date
latest_flight_date_li = ["2023" + s for s in ["0503","0518", "0530", "0612", "0627", "0718"]]
flight_date_li = ['202304171517', '202305031115', '202305181131', '202305301156', '202306121122', '202306271027']
mul_flight_date_li = ['202305031115', '202305181131', '202305301127', '202306121122', '202306271027']
latest_flight_date_li = ["2023" + s for s in ["0503","0518", "0530", "0612", "0627", "0718"]] 

Label_names = [
    "ID4human",
    "Origin",
    "Density",
    "block",
    "subblock",
    # "ridge",
    # "plot",
    "HarvTime"
]

# Calculate Days after Sowing
reference_date = pd.to_datetime('20230330', format='%Y%m%d')
samp_date_dic ={"1":"20230519", "2":"20230601", "3":"20230615", "4":"20230629", "5":"20230719"}
samp_DAS_dic = {str(s):(dt.datetime.strptime(samp_date_dic[str(s)],'%Y%m%d') - reference_date).days for s in range(1,6)}

# Feature names
RGB_Pheno_names = ["PH_90", "CC_rate", "PV_ave", "ExG_VI", "VARI_VI", "GRVI_VI"]
Mul_VI_names = ["NDVI", "GNDVI", "NDRE"]
Mul_Pheno_names = [s + "_median" for s in Mul_VI_names]
UAV_pheno_names = RGB_Pheno_names + Mul_Pheno_names


## Execution

In [21]:
time_now = dt.datetime.now().strftime("%Y%m%d%H%M")
print(time_now)

model_name_li = ["RF"]  
pred_df = df.copy() 

i_t = 0 
target_name = "Wei_all_per_m2"

for harv_i in tqdm(range(1,5)): 
    harv_now = str(harv_i)
    print(harv_now)
    resu_df4 = K10foldCVwithPipe5(time_now = time_now, 
                                  rep_num=3, 
                                  harv_now = harv_now, 
                                  model_name_li=model_name_li,
                                  latest_flight_date_li = latest_flight_date_li, 
                                  df = pred_df, 
                                  target_name= target_name, 
                                  seed=seed, 
                                  demo = False,
                                  do_cv= True, 
                                  do_savefig= True)

202605181322


  0%|          | 0/4 [00:00<?, ?it/s]

1
0 RGB 25 random 1


KeyError: "['subblock'] not in index"

## Output Formatting for Growth Curve Modeling

In [ ]:
# Specify when to use the trained model set
modeltime_now =  CV_modeltime_now # "%Y%m%d%H%M"
CV_modeltime_now = model_timestamp_config_df.loc["RandomCV", year_now]

In [ ]:
modeltime_now_dir =  results_dir / f"merge_pred_df_melt_{modeltime_now}" 
os.makedirs(modeltime_now_dir, exist_ok=True)

In [ ]:
def get_stra_names(modeltime_now):
    for harv_now in range(1,5) :    

        model_output_dir = results_dir / f"CV10_harv{str(harv_now)}" / f"K10foldCVwithPipe5_{modeltime_now}"
        output_li = os.listdir(model_output_dir)
        csv_names = [s for s in output_li if (s.startswith("pred_df_proto") & s.endswith(".csv"))] 
        stra_names = ["bloCV" + s.split("bloCV")[1].split("Harv")[0] + "ImgType" + s.split("ImgType")[1] for s in csv_names]
        
    return stra_names

In [ ]:
stra_names = get_stra_names(modeltime_now)
stra_names_df = pd.DataFrame(stra_names, columns=["stra_names"])
stra_names_df.to_csv(modeltime_now_dir / "stra_names_df.csv")
stra_names_df

In [ ]:
def make_merge_pred_df(stra_name_now, pri = False): 
    harv_li = [str(s) for s in [1,2,3,4] ] 
    
    for harv_i in range(len(harv_li)):
        model_output_dir = results_dir / f"CV10_harv{harv_li[harv_i]}"/ f"K10foldCVwithPipe5_{modeltime_now}" 
        search_csv_name_now = f"{stra_name_now}" 
        search_files = [s for s in os.listdir(model_output_dir) if (set(search_csv_name_now.split("_")).issubset(s.split("_")))&(s.endswith(".csv"))&s.startswith("pred_df_proto")]# <---<---<---<---<---<---<---<---<---<---<---<---<---<---<---
        csv_name_now = list(search_files)[0]
        
        if pri == True:
            print(csv_name_now)

        if harv_i==0:
            proto_now_df = pd.read_csv(model_output_dir / csv_name_now, index_col=0).drop("pred_DAS", axis = 1)
            merge_pred_df = proto_now_df
            
        else:
            proto_now_df = pd.read_csv(model_output_dir / csv_name_now, index_col=0).drop("pred_DAS", axis = 1)
            merge_pred_df =  pd.merge(merge_pred_df, proto_now_df, how = "left")
            
    merge_pred_df = df_formatting(merge_pred_df, Label_names)
    merge_pred_df = merge_pred_df[merge_pred_df["HarvTime"] == "5"] 
    pred_names_li = [s for s in merge_pred_df.columns if s.endswith("_pred")] 
    merge_pred_df = merge_pred_df.loc[:, Label_names + [target_name] + pred_names_li]
    
    return merge_pred_df

In [ ]:
for (stra_i,) in tqdm(itertools.product(range(len(stra_names)))):        
    stra_name_now = stra_names[stra_i]
    merge_pred_df = make_merge_pred_df(stra_name_now)  

In [ ]:
def make_merge_pred_df_melt(merge_pred_df, target_name, pri = False):
    merge_pred_df_melt = pd.melt(merge_pred_df, id_vars=Label_names)
    for i in range(1,6):
        if pri == True:
            print(i)
        if i == 5:
            merge_pred_df_melt.loc[merge_pred_df_melt["variable"]==target_name, "pred_DAS"] = samp_DAS_dic[str(i)]
        else:
            merge_pred_df_melt.loc[merge_pred_df_melt["variable"].str.contains("Harv-" + str(i)), "pred_DAS"] = samp_DAS_dic[str(i)]  
    merge_pred_df_melt["index"] = target_name
    merge_pred_df_melt.sort_values("pred_DAS", inplace=True)

    return merge_pred_df_melt

In [ ]:
merge_pred_df_melt = make_merge_pred_df_melt(merge_pred_df, target_name)
merge_pred_df_melt

In [ ]:
merge_pred_df_melt.to_csv(results_dir / f"merge_pred_df_melt_{modeltime_now}.csv")

In [ ]:
stra_names = get_stra_names(modeltime_now)
for stra_i in tqdm(range(len(stra_names))):
    stra_name_now = stra_names[stra_i]
    merge_pred_df = make_merge_pred_df(stra_name_now)
    merge_pred_df_melt = make_merge_pred_df_melt(merge_pred_df, target_name)
    
    merge_pred_df_melt.to_csv(modeltime_now_dir / f"{stra_name_now}.csv")

# Block-wise Cross-Validation

In [ ]:
def K10foldBlockCVwithPipe4(time_now, rep_num, harv_now, model_name_li, df, target_name, seed, latest_flight_date_li,
                            demo = False, do_cv=True, do_savefig=True, debag = False): 
    # Setting the parameter   
    strategy_now = inspect.currentframe().f_code.co_name 
    ran_num = 0
    ii = 0 
    target_min_max_li = [[0],[8000]] 
    target_HaT_min_max_li = [[0, 0,1000,2000,3000,4000],
                             [1000, 1500,3000, 5500, 7000, 8000]] 
    # Picking up the harvest time data
    HarvTime_now = int(harv_now)
    harv_now_df = df[df["HarvTime"] == str(HarvTime_now)].copy()
    harv_now_df.reset_index(drop=True, inplace=True)

    # Creating data frame to store the results of model performance evaluation
    model_eva_li = ["r", "R2", "RMSE"] 
    resu_df_col_names = ["model_fullname", 
                         "HarvTime", 
                         "Flight_date", 
                         "img_type", 
                         "samp_rate", 
                         "samp_method", 
                         "rep", 
                         "prediction_modes"] + model_eva_li 
    resu_df = pd.DataFrame(index = [], columns=resu_df_col_names) 
    
    # Save dir
    str_dt_now = time_now 
    if demo == True:
        base_model_dir = midstream_dir / f"CV10_harv{harv_now}" / f"{strategy_now}_{str_dt_now}"
    else:
        base_model_dir = results_dir / f"CV10_harv{harv_now}" / f"{strategy_now}_{str_dt_now}"
    
    print(base_model_dir)
    os.makedirs(base_model_dir, exist_ok=True)

    # ============================================
    # Flight Combination Settings
    # ============================================
    latest_flight_date = latest_flight_date_li[HarvTime_now-1]
    before_flight_datetime_li = [s for s in flight_date_li if s[:8] in (latest_flight_date_li[:(HarvTime_now+1)])] 
    input_num_li = [list(itertools.combinations([i for i in range(0,HarvTime_now+1)],s))for s in range(1, HarvTime_now+2)]
    input_num_li = [s1 for s2 in input_num_li for s1 in s2]
    
    gap = 50 
    per_li = [s for s in range(gap,100+gap,gap)]
    
    for input_num_now, img_type, per_i, ran_or_kme in itertools.product(range(len(input_num_li)),["RGB", "Full"], range(len(per_li)),["random", "kmeans"]):
        if demo == True:
            if img_type != "RGB":
                continue
            if ran_or_kme != "kmeans":
                continue

        input_num_name_now = "".join([str(s) for s in list(input_num_li[input_num_now])]) 
        input_comb_date_now = [latest_flight_date_li[s] for s in list(input_num_li[input_num_now])] 
        input_comb_now = [s for s in flight_date_li if s[:8] in (input_comb_date_now)] 

        ## Setting the input
        RGB_input_comb_now = [s for s in flight_date_li if s[:8] in (input_comb_date_now)]
        RGB_input_UAV_cols = [[ s1 + "_" + s for s1 in RGB_Pheno_names] for s in RGB_input_comb_now] 
        RGB_input_UAV_cols = [x for row in RGB_input_UAV_cols for x in row] 
        
        mul_input_comb_now = [s for s in mul_flight_date_li if s[:8] in (input_comb_date_now)]
        mul_input_UAV_cols = [[ s1 + "_" + s for s1 in  Mul_Pheno_names] for s in mul_input_comb_now] 
        mul_input_UAV_cols = [x for row in mul_input_UAV_cols for x in row]
        
        if img_type == "RGB": 
            input_UAV_cols = RGB_input_UAV_cols 

        elif img_type == "Multi": 
            input_UAV_cols = mul_input_UAV_cols 

        elif img_type == "Full": 
            input_UAV_cols = RGB_input_UAV_cols + mul_input_UAV_cols 
            
        ## Replication 
        ran_num +=1
        for rep_now in range(1, rep_num + 1):

            pred_df = df.copy() 
            proto_pred_df = df.copy() 
            print(f"model setting are {input_num_now}, {img_type}, {per_li[per_i]}, {ran_or_kme}, {rep_now}")

            ## Setting the seed
            random.seed(seed)
            rep_dict = dict(zip([str(s) for s in range(1,rep_num + 1)], random.sample(range(1,100), k = rep_num))) 

            ## Labeling 10fold
            CV_k = 10             
            CV_k_blo = int(CV_k/2) 
            kf_blo = KFold(n_splits = CV_k_blo, shuffle = True, random_state=rep_dict[str(rep_now)]) 
            CV_alpha_li = list(string.ascii_uppercase[0:CV_k])
            
            for blo_now in [1,2]:
                blo_now_harv_now_df = harv_now_df[harv_now_df["block"] == f"T{str(blo_now)}"].copy()  
                blo_now_harv_now_df.reset_index(drop=True, inplace=True)
                
                train_group_index_li = []
                test_group_index_li = []
                
                for train_i, test_i in kf_blo.split(blo_now_harv_now_df):
                    train_group_index_li += [train_i.tolist()]
                    test_group_index_li += [test_i.tolist()]
                    
                CV_alpha_li_blo = list(string.ascii_uppercase[((blo_now-1)*CV_k_blo):(blo_now*CV_k_blo)])
                
                for group_num_now, group_alp_now in zip(range(0,CV_k_blo),CV_alpha_li_blo ): 
                    blo_now_harv_now_df.loc[test_group_index_li[group_num_now], "Group" + str(harv_now) + "_" + str(rep_now)] = group_alp_now
                    
                    
                if blo_now == 1:
                    temp_df = blo_now_harv_now_df

                else:
                    temp_df = pd.concat([temp_df,blo_now_harv_now_df])
            
            temp_df.reset_index(drop=True, inplace=True)
            
            if ("Group" + str(harv_now) + "_" + str(rep_now)) not in harv_now_df.columns:
                harv_now_df = pd.merge(harv_now_df,
                                       temp_df[["ID4human", "Group" + str(harv_now) + "_" + str(rep_now)]],
                                       how = "left",
                                       on = "ID4human")

            ## Setting the data frame of whole data
            df_now = harv_now_df[Label_names + ["Group" + str(harv_now) + "_" + str(rep_now)] + input_UAV_cols + [target_name]] 
            X = df_now[Label_names + ["Group" + str(harv_now) + "_" + str(rep_now)] + input_UAV_cols]
            y = df_now[Label_names + ["Group" + str(harv_now) + "_" + str(rep_now)] + [target_name]]
            if demo == True: print(f"input_UAV_cols is {input_UAV_cols}") 
            if demo == True: print(f"target_name is {target_name}")

            ## Setting the data frame for estimation
            pheno_df_now = pred_df[pred_df["HarvTime"] >= str(HarvTime_now)][Label_names + input_UAV_cols + [target_name]]  
            base_X_pheno = pheno_df_now[input_UAV_cols] 

            y_pheno = pheno_df_now[target_name]
            
            # ============================================
            # Model Training Methods and Parameter Settings
            # ============================================
            
            ## Regressor 
            regressor_dic = {"RF":RandomForestRegressor(random_state=rep_dict[str(rep_now)])}
            ## Feature selection
            ft_sele_now = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state= rep_dict[str(rep_now)]),
                                          threshold="median")
            pipe = Pipeline([("scaler", StandardScaler()), 
                             ("feature_selection",ft_sele_now),
                             ("regressor",[regressor_dic["RF"]])]) 

            param_grid = [{"regressor":[regressor_dic['RF']], 
                 "scaler":[StandardScaler()], 
                 "feature_selection":[ft_sele_now],
                 "regressor__n_estimators":[10,100,1000],
                 "regressor__max_depth":[5,10,15, 20, None]}]

            model_fullname_li = ["Harv-" + str(HarvTime_now),
                                 "Model-" + "RF",                                 
                                 "UAVinput-"+ input_num_name_now, 
                                 "ImgType-" + img_type, 
                                 "SampRate-" + str(per_li[per_i]),
                                 "SampMetho-" + ran_or_kme, 
                                 "Rep-" + str(rep_now)]
            model_fullname_ori = "_".join(model_fullname_li)
            
            # ============================================
            # Proto model
            # ============================================            
            debag and print(f"-the number of plots in original data is ：",len(X))
            for bloCV_now in ["bloAll", "blo1","blo2"]:
                print(bloCV_now)
                blo_model_fullname = "bloCV-" + bloCV_now + "_" + model_fullname_ori 
                proto_model_fullname = "proto_" + blo_model_fullname

                ## Dataset of explanatory variables for proto models
                if bloCV_now == "bloAll":
                    proto_X_train = X[input_UAV_cols]
                    proto_y_train = y[target_name]
                    CV_k_proto = CV_k 
                    
                else:
                    proto_X_train = X.loc[X["block"] == f"T{str(blo_now)}", input_UAV_cols] 
                    proto_y_train = y.loc[y["block"] == f"T{str(blo_now)}", target_name]
                    CV_k_proto = CV_k_blo
                    
    
                tra_rate = (CV_k_blo-1)/CV_k_proto 
                debag and print("tra_rate:",tra_rate)

                ## Check the sampling strategy
                if ran_or_kme == "random":
                    proto_X_train = proto_X_train.sample(frac=(tra_rate),
                                                         random_state = ran_num).sort_index()
                    X_train = proto_X_train.sample(frac=((per_li[per_i]/100)), 
                                                   random_state = ran_num).sort_index() 
                    y_train = proto_y_train[X_train.index.values.tolist()].sort_index()

                elif ran_or_kme == "kmeans":
                    proto_X_train = proto_X_train.sample(frac=(tra_rate),
                                                         random_state = ran_num + 1).sort_index()
                    kmeans = KMeans(n_clusters=int((len(proto_X_train )*(per_li[per_i]/100))),
                                    random_state= (ran_num + 1), 
                                    n_init="auto")

                    _ = kmeans.fit(proto_X_train)

                    closest, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, proto_X_train)
                    X_train =  proto_X_train.iloc[closest].sort_index()
                    y_train = proto_y_train[X_train.index.values.tolist()].sort_index()


                debag and print(f"-in {bloCV_now}, the number of plots in training：",len(X_train))    

                proto_grid = GridSearchCV(pipe, param_grid, scoring = "neg_root_mean_squared_error", cv=4, n_jobs= 10, verbose=0) 
                _ = proto_grid.fit(X_train, y_train) ; 

                with open(base_model_dir / f"{proto_model_fullname}.pickele", mode="wb") as f: 
                    pickle.dump(proto_grid, f)

                with open(base_model_dir / f"{proto_model_fullname}.pickele", mode="rb") as f: 
                    proto_grid = pickle.load(f)

                ## Estimation using proto model
                proto_pred_df = df.copy()
                proto_y_pheno_pred_df = pheno_df_now.copy() 
                proto_y_pheno_pred = proto_grid.predict(base_X_pheno) 
                proto_y_pheno_pred_df[proto_model_fullname + "_pred"] = proto_y_pheno_pred 
                proto_pred_df = pd.merge(proto_pred_df, proto_y_pheno_pred_df, how="left") 
                proto_pred_df["pred_DAS"] = samp_DAS_dic[harv_now]

                proto_pred_df.to_csv(base_model_dir / f"pred_df_{proto_model_fullname}.csv") 
                proto_pred_df = pd.read_csv(base_model_dir  / f"pred_df_{proto_model_fullname}.csv", index_col=0)
                proto_pred_df = df_formatting(proto_pred_df, Label_names)
                # ============================================
                # CV
                # ============================================
                for CV_now in (CV_alpha_li): 
                    test_CVgroup_now = [CV_now] 
                    debag and print(test_CVgroup_now)
                
                    if bloCV_now == "bloAll": # Selectable from anywhere except the test group
                        train_CVgroup_now = [s for s in CV_alpha_li if s != CV_now]
                        CV_k_CVblo = CV_k - 1 
                        debag and print(train_CVgroup_now)
                    
                    else: 
                        CV_alpha_li_blo = list(string.ascii_uppercase[((int(bloCV_now[-1])-1)*CV_k_blo):(int(bloCV_now[-1])*CV_k_blo)])

                        train_CVgroup_now = [s for s in CV_alpha_li_blo if s != CV_now] 
                        if len(train_CVgroup_now) == 4:
                            CV_k_CVblo = CV_k_blo - 1
                            
                        else:
                            CV_k_CVblo = CV_k_blo
                        debag and print(train_CVgroup_now)
                    
                    base_X_train = X.loc[X["Group" + str(harv_now) + "_" + str(rep_now)].isin(train_CVgroup_now),input_UAV_cols]
                    base_X_test = X.loc[X["Group" + str(harv_now) + "_" + str(rep_now)].isin(test_CVgroup_now),input_UAV_cols]

                    base_y_train = y.loc[y["Group" + str(harv_now) + "_" + str(rep_now)].isin(train_CVgroup_now),target_name]
                    y_test = y.loc[y["Group" + str(harv_now) + "_" + str(rep_now)].isin(test_CVgroup_now),target_name]
                    
                    seed_4samp = int((rep_dict[str(rep_now)] + 100) * per_li[per_i]) + ran_num
                    
                    tra_rate = (CV_k_blo-1)/CV_k_CVblo 
                    debag and print("CV_k_blo:",CV_k_blo)
                    debag and print("CV_k_CVblo:",CV_k_CVblo)
                    debag and print("tra_rate:",tra_rate)
                    
                    if ran_or_kme == "random":
                        base_X_train = base_X_train.sample(frac=(tra_rate),
                                                         random_state = seed_4samp).sort_index()
                        X_train = base_X_train.sample(frac=(per_li[per_i]/100), random_state = seed_4samp).sort_index()
                        y_train = base_y_train[X_train.index.values.tolist()].sort_index()

                    elif ran_or_kme == "kmeans":
                        base_X_train = base_X_train.sample(frac=(tra_rate),
                                                         random_state = seed_4samp + 1).sort_index()
                        kmeans = KMeans(n_clusters=int((len(base_X_train )*(per_li[per_i]/100))),
                                        random_state= (seed_4samp + 1),
                                        n_init="auto")

                        _ = kmeans.fit(base_X_train)

                        closest, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, base_X_train)
                        X_train =  base_X_train.iloc[closest].sort_index()
                        y_train = base_y_train[X_train.index.values.tolist()].sort_index()

                    X_test = base_X_test 
                    y_test = y_test 

                    model_fullname = f"{blo_model_fullname}_TestGroup-{CV_now}" 

                    debag and print(f"-{bloCV_now},{CV_now}, the number of plots：",len(X_train)) 
                    
                    ## Train the model
                    if do_cv == True:
                        grid = GridSearchCV(pipe, param_grid, scoring = "neg_root_mean_squared_error", cv=4, n_jobs= 10, verbose=0) 
                        _ = grid.fit(X_train, y_train) ; 

                        with open(base_model_dir / f"{model_fullname}.pickele", mode="wb") as f: 
                            pickle.dump(grid, f)

                        with open(base_model_dir / f"{model_fullname}.pickele", mode="rb") as f: 
                            grid = pickle.load(f)

                        y_pheno_pred_df = pheno_df_now.copy() 
                        y_pheno_pred = grid.predict(base_X_pheno) 
                        y_pheno_pred_df[model_fullname + "_pred"] = y_pheno_pred 

                        pred_df = pd.merge(pred_df, y_pheno_pred_df, how="left") 
                        
                        ## Estimation
                        y_train_pred = grid.predict(X_train) 
                        y_test_pred = grid.predict(X_test) 
                        output_ndim = y_train_pred.ndim 
                        if output_ndim == 2: 
                            y_train_pred = np.array([s1 for s2 in y_train_pred.tolist() for s1 in s2])
                            y_test_pred = np.array([s1 for s2 in y_test_pred.tolist() for s1 in s2])

                        ymin_now = target_min_max_li[0][i_t] 
                        ymax_now = target_min_max_li[1][i_t] 

                        CV_train_df = pd.DataFrame(
                            {"y_train":y_train,
                             "y_train_pred":y_train_pred})
                        CV_train_df.to_csv(base_model_dir / f"train_df_{model_fullname}.csv")
                        CV_train_df = pd.read_csv(base_model_dir / f"train_df_{model_fullname}.csv", index_col=0)

                        CV_test_df = pd.DataFrame(
                            {"y_test":y_test,
                             "y_test_pred":y_test_pred})
                        CV_test_df.to_csv(base_model_dir / f"test_df_{model_fullname}.csv")
                        CV_test_df = pd.read_csv(base_model_dir / f"test_df_{model_fullname}.csv", index_col=0)

            # Summalize of block CV
            if do_cv == True:   
                  
                key_df = df_now.loc[:, ["ID4human", "Group" + str(harv_now) + "_" + str(rep_now)]]  
                
                pred_df_now = pd.merge(pred_df, y_pheno_pred_df, how="left") 
                pred_df = pd.merge(pred_df, key_df, how = "left", on = "ID4human")
                pred_df.to_csv(base_model_dir / f"pred_df_{model_fullname_ori}.csv") 
                
                pred_df = pd.read_csv(base_model_dir / f"pred_df_{model_fullname_ori}.csv", index_col=0) 
    

                prediction_modes_li = ['group_within', 'group_between', 'overall']
                for prediction_modes_now in prediction_modes_li:
                    pred_alpha_blo1_li = [f"{chr(c)}_pred" for c in range(ord("A"), ord("F"))]
                    pred_alpha_blo2_li = [f"{chr(c)}_pred" for c in range(ord("F"), ord("K"))]
                    
                    if prediction_modes_now == 'group_within':

                        filtered_columns_blo1_blo1 = [col for col in pred_df.columns if ("blo1" in col ) and any(suf in col for suf in pred_alpha_blo1_li)]
                        filtered_columns_blo2_blo2 = [col for col in pred_df.columns if ("blo2" in col ) and any(suf in col for suf in pred_alpha_blo2_li)] 
                        filtered_columns = filtered_columns_blo1_blo1 + filtered_columns_blo2_blo2 
                    
                    elif prediction_modes_now == 'group_between':

                        filtered_columns_blo1_blo2 = [col for col in pred_df.columns if ("blo1" in col ) and any(suf in col for suf in pred_alpha_blo2_li)] 
                        filtered_columns_blo2_blo1 = [col for col in pred_df.columns if ("blo2" in col ) and any(suf in col for suf in pred_alpha_blo1_li)] 
                        filtered_columns = filtered_columns_blo1_blo2 + filtered_columns_blo2_blo1
                    
                    elif prediction_modes_now == 'overall':
                        filtered_columns = [col for col in pred_df.columns if ("bloAll" in col)] 
                       
                    pred_df_now = pred_df[Label_names + target_name_list + filtered_columns + [f"Group{str(harv_now)}_{str(rep_now)}"]]
                    pred_df_now = df_formatting(pred_df_now, Label_names)

                    x_all, y_all = all_test_dataframe(pred_df_now,CV_alpha_li, harv_now,rep_now, target_name)
                    if do_savefig == True:
            
                        fig = all_test_plot(x_all, y_all,target_HaT_min_max_li[0][HarvTime_now-1],target_HaT_min_max_li[1][HarvTime_now-1])
                        fig.savefig(base_model_dir / f"scatplo_{model_fullname_ori}.png")
                        plt.show(fig)
                        plt.close(fig)

                    ## Creating a data frame of model evaluation metrics                
                    r = pd.Series(x_all.to_list()).corr(pd.Series(y_all.tolist()))
                    R2 = r2_score(x_all, y_all)
                    RMSE = mean_squared_error(x_all, y_all, squared=False)
                    RMSE_rounded = float(remove_exponent(Decimal(RMSE).quantize(Decimal("0.01"), ROUND_HALF_UP)))

                    resu_li_now = [model_fullname_ori, HarvTime_now, input_num_now, img_type, per_li[per_i], ran_or_kme, rep_now, prediction_modes_now] 
                    resu_li_now = resu_li_now + [r, R2, RMSE] 
                    resu_df.loc[ii] = resu_li_now 
                    ii += 1
          
        resu_df.to_csv(base_model_dir /  f"resu_df_{harv_now}.csv" )
    
    if do_cv == False:
        resu_df = "None"
    return resu_df 

## Run block CV

In [ ]:
time_now = dt.datetime.now().strftime("%Y%m%d%H%M") 
print(time_now)

model_name_li = ["RF"]    
pred_df = df.copy()

i_t = 0 
target_name_list = ["Wei_all_per_m2"]
target_name = "Wei_all_per_m2"

for harv_i in tqdm(range(1,5)): 
    harv_now = str(harv_i)
    print(harv_now)
    resu_df4 = K10foldBlockCVwithPipe4(time_now = time_now, 
                                       rep_num= 3,
                                       harv_now = harv_now, 
                                       model_name_li=model_name_li,
                                       latest_flight_date_li = latest_flight_date_li, 
                                       df = pred_df, 
                                       target_name= target_name, 
                                       seed=seed, 
                                       demo = False,
                                       do_cv=True, 
                                       do_savefig=False, 
                                       debag=False)